In [39]:
import xml.etree.ElementTree as ET
import json, random, os
from collections import Counter
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
!pip -q install python-Levenshtein==0.23.0
import Levenshtein as Lev

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


**Парсинг XML-файла и извлечение данных**

XML-файл разбирается на предложения, слова и аллофоны.
Для каждого слова извлекаются также дополнительные признаки: паузы, часть речи и позиция в предложении.

In [40]:
XML_PATH = "/kaggle/input/jeromedata/Jerome.Result.xml"

tree = ET.parse(XML_PATH)
root = tree.getroot()

def extract_allophones(word_el):
    return [a.attrib.get("ph", "").strip() for a in word_el.findall(".//allophone") if a.attrib.get("ph")]

def get_pos(word_el):
    di = word_el.find(".//dictitem")
    if di is None:
        return "UNKPOS"
    for key in ["part_of_speech", "subpart_of_speech", "pos"]:
        if key in di.attrib:
            return di.attrib[key]
    return "UNKPOS"

pairs = []
for s_idx, sent in enumerate(root.findall(".//sentence")):
    children = list(sent)
    for i, ch in enumerate(children):
        if ch.tag != "word":
            continue
        word = ch.attrib.get("original", "")
        allos = extract_allophones(ch)
        if not allos:
            continue
        pos = get_pos(ch)
        pause_before = (i - 1 >= 0 and children[i - 1].tag == "pause")
        pause_after = (i + 1 < len(children) and children[i + 1].tag == "pause")
        feats = {
            "pause_before": int(pause_before),
            "pause_after": int(pause_after),
            "pos": pos,
            "word_pos_from_start": i,
            "word_pos_from_end": len(children) - i - 1,
        }
        pairs.append((word, allos, feats))

print("Пример:", pairs[0])
print("Всего слов:", len(pairs))


Пример: ('Джером', ['d', 'zh', 'y1', 'r', 'o0', 'm'], {'pause_before': 0, 'pause_after': 1, 'pos': '1', 'word_pos_from_start': 2, 'word_pos_from_end': 8})
Всего слов: 42441


**Подготовка токенов и построение словарей**

Здесь создаются токены для входных и выходных последовательностей, а также словари для кодирования символов и аллофонов.

In [ ]:
BOS, EOS, PAD = "<s>", "</s>", "<pad>"

def build_vocab(items, min_freq=1):
    cnt = Counter()
    for seq in items:
        cnt.update(seq)
    itos = [PAD, BOS, EOS] + [t for t, f in cnt.items() if f >= min_freq]
    stoi = {t: i for i, t in enumerate(itos)}
    return stoi, itos

def encode(seq, stoi):
    return [stoi[BOS]] + [stoi.get(t, stoi.get("<unk>", len(stoi))) for t in seq] + [stoi[EOS]]

def decode(ids, itos):
    return [itos[i] for i in ids if itos[i] not in (BOS, EOS, PAD)]

def grapheme_tokens(word, feats):
    chars = list(word)
    feat_toks = [
        f"<PB{feats['pause_before']}>",
        f"<PA{feats['pause_after']}>",
        f"<POS={feats['pos']}>",
        f"<WFS={feats['word_pos_from_start']}>",
        f"<WFE={feats['word_pos_from_end']}>",
    ]
    return feat_toks + chars

X_tokens = [grapheme_tokens(w, f) for (w, a, f) in pairs]
Y_tokens = [a for (w, a, f) in pairs]

src_stoi, src_itos = build_vocab(X_tokens)
tgt_stoi, tgt_itos = build_vocab(Y_tokens)

print("Размер словаря входа:", len(src_itos))
print("Размер словаря выхода:", len(tgt_itos))


**Подготовка датасета и функции collate**

Здеьс определяется класс для хранения данных и функция, которая формирует батчи для обучения.

In [42]:
class G2ADataset(Dataset):
    def __init__(self, pairs, src_stoi, tgt_stoi):
        self.data = [(grapheme_tokens(w, f), a, f) for w, a, f in pairs]
        self.src_stoi = src_stoi
        self.tgt_stoi = tgt_stoi

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def collate(batch):
    src_ids = [encode(x[0], src_stoi) for x in batch]
    tgt_ids = [encode(x[1], tgt_stoi) for x in batch]
    src_max = max(len(s) for s in src_ids)
    tgt_max = max(len(s) for s in tgt_ids)
    src_pad = [s + [0] * (src_max - len(s)) for s in src_ids]
    tgt_pad = [t + [0] * (tgt_max - len(t)) for t in tgt_ids]
    return torch.tensor(src_pad), torch.tensor(tgt_pad)


**Разделение данных на train, validation и test**

In [30]:
random.shuffle(pairs)
n = len(pairs)
train = pairs[:int(0.8 * n)]
val = pairs[int(0.8 * n):int(0.9 * n)]
test = pairs[int(0.9 * n):]

train_dl = DataLoader(G2ADataset(train, src_stoi, tgt_stoi), batch_size=256, shuffle=True, collate_fn=collate)
val_dl = DataLoader(G2ADataset(val, src_stoi, tgt_stoi), batch_size=256, shuffle=False, collate_fn=collate)
test_dl = DataLoader(G2ADataset(test, src_stoi, tgt_stoi), batch_size=256, shuffle=False, collate_fn=collate)


**Определение Transformer-модели**

In [43]:
class Seq2SeqTransformer(nn.Module):
    def __init__(self, num_encoder_layers, num_decoder_layers, emb_size, nhead,
                 src_vocab_size, tgt_vocab_size, dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.src_tok_emb = nn.Embedding(src_vocab_size, emb_size, padding_idx=0)
        self.tgt_tok_emb = nn.Embedding(tgt_vocab_size, emb_size, padding_idx=0)
        self.pos_encoder = nn.Embedding(512, emb_size)
        self.pos_decoder = nn.Embedding(512, emb_size)
        self.transformer = nn.Transformer(
            d_model=emb_size, nhead=nhead,
            num_encoder_layers=num_encoder_layers, num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True
        )
        self.generator = nn.Linear(emb_size, tgt_vocab_size)

    def forward(self, src, tgt):
        src_pos = torch.arange(0, src.size(1), device=src.device).unsqueeze(0)
        tgt_pos = torch.arange(0, tgt.size(1), device=src.device).unsqueeze(0)
        src_emb = self.src_tok_emb(src) + self.pos_encoder(src_pos)
        tgt_emb = self.tgt_tok_emb(tgt) + self.pos_decoder(tgt_pos)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(src.device)
        out = self.transformer(src_emb, tgt_emb, tgt_mask=tgt_mask)
        return self.generator(out)


**Обучение модели**

In [44]:
model = Seq2SeqTransformer(3, 3, 256, 8, len(src_itos), len(tgt_itos)).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

def run_epoch(dl, train=True):
    model.train(train)
    total_loss = 0.0
    for src, tgt in dl:
        src, tgt = src.to(device), tgt.to(device)
        dec_inp = tgt[:, :-1]
        gold = tgt[:, 1:].contiguous().view(-1)
        logits = model(src, dec_inp).contiguous().view(-1, len(tgt_itos))
        loss = criterion(logits, gold)
        if train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dl)

best_val = 1e9
for epoch in range(20):
    tr = run_epoch(train_dl, True)
    vl = run_epoch(val_dl, False)
    if vl < best_val:
        best_val = vl
        torch.save(model.state_dict(), "best.pt")
    print(f"Epoch {epoch+1}: train {tr:.3f} | val {vl:.3f}")


Epoch 1: train 1.795 | val 0.432
Epoch 2: train 0.355 | val 0.204
Epoch 3: train 0.224 | val 0.155
Epoch 4: train 0.180 | val 0.137
Epoch 5: train 0.151 | val 0.121
Epoch 6: train 0.133 | val 0.109
Epoch 7: train 0.119 | val 0.102
Epoch 8: train 0.109 | val 0.099
Epoch 9: train 0.101 | val 0.097
Epoch 10: train 0.095 | val 0.095
Epoch 11: train 0.091 | val 0.088
Epoch 12: train 0.085 | val 0.089
Epoch 13: train 0.080 | val 0.090
Epoch 14: train 0.077 | val 0.090
Epoch 15: train 0.074 | val 0.084
Epoch 16: train 0.070 | val 0.090
Epoch 17: train 0.068 | val 0.083
Epoch 18: train 0.064 | val 0.088
Epoch 19: train 0.062 | val 0.082
Epoch 20: train 0.061 | val 0.084


**Расчет метрик**

In [45]:
model.load_state_dict(torch.load("best.pt", map_location=device))
model.eval()

def greedy_decode(word, feats, max_len=80):
    src = torch.tensor([encode(grapheme_tokens(word, feats), src_stoi)], device=device)
    tgt = torch.tensor([[tgt_stoi["<s>"]]], device=device)
    for _ in range(max_len):
        logits = model(src, tgt)
        next_id = logits[:, -1, :].argmax(-1)
        tgt = torch.cat([tgt, next_id.unsqueeze(1)], dim=1)
        if next_id.item() == tgt_stoi["</s>"]:
            break
    return decode(tgt[0].tolist(), tgt_itos)

def seq_metrics(pred, gold):
    tp = sum(1 for p, g in zip(pred, gold) if p == g)
    precision = tp / max(len(pred), 1)
    recall = tp / max(len(gold), 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-9)
    return precision, recall, f1

def wrr_word(pred, gold):
    lev = Lev.distance(" ".join(pred), " ".join(gold))
    denom = max(len(" ".join(gold)), 1)
    return 1.0 - (lev / denom)


**Оценка модели и сохранение результатов**

In [46]:
P = R = F = WRR = 0.0
results_json = {"text": []}

sentence = []
sentence_index = 0
word_counter = 0
words_per_sentence = 10  

for idx, (w, a, f) in enumerate(tqdm(test, total=len(test))):
    pr = greedy_decode(w, f)
    p, r, f1 = seq_metrics(pr, a)
    wr = wrr_word(pr, a)
    P += p; R += r; F += f1; WRR += wr

    sentence.append({"word": w, "allophone": pr})
    word_counter += 1

    if word_counter >= words_per_sentence or idx == len(test) - 1:
        results_json["text"].append({
            "sentence_index": sentence_index,
            "words": sentence
        })
        sentence_index += 1
        sentence = []
        word_counter = 0

n = len(test)
print({
    "Precision": P/n,
    "Recall": R/n,
    "F1": F/n,
    "WRR": WRR/n
})

with open("results_correct_format.json", "w", encoding="utf-8") as f:
    json.dump(results_json, f, ensure_ascii=False, indent=4)

print("Сохранено: results_correct_format.json")


  0%|          | 0/4245 [00:00<?, ?it/s]

{'Precision': 0.8919078676275722, 'Recall': 0.9012074211653295, 'F1': 0.894746147469062, 'WRR': 0.9023817877230683}
Сохранено: results_correct_format.json
